# FreshRetailNet-50K — Data Ingestion & Preprocessing for MMF

## Dataset Overview

**FreshRetailNet-50K** is an open benchmark dataset for perishable grocery demand forecasting,
released by [Dingdong Inc.](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K)
It contains **50,000 store-product time series** of daily sales data from **898 stores** across
**18 major cities**, covering **865 perishable SKUs** over a 90-day period (Mar–Jun 2024).

| Property | Value |
|---|---|
| **Source** | [HuggingFace — Dingdong-Inc/FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K) |
| **License** | **Creative Commons Attribution 4.0 International (CC BY 4.0)** |
| **Rows** | ~4.85M (4.5M train + 350K eval) |
| **Time Series** | 50,000 store-product combinations |
| **Date Range** | 90 consecutive days (Mar 28 – Jun 25, 2024) |
| **Stores** | 898 across 18 cities |
| **Products** | 865 perishable SKUs |
| **External Regressors** | Discount, holiday, promotions, precipitation, temperature, humidity, wind |

### License & Attribution

This dataset is licensed under **CC BY 4.0**, which permits sharing, adaptation, and commercial use — including publication in blog posts — provided that appropriate credit is given.

> **Citation:** Dingdong Inc. (2025). FreshRetailNet-50K: A Stockout-Annotated Censored Demand Dataset for Latent Demand Recovery and Forecasting in Fresh Retail. https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K

### Why This Dataset for MMF?

- **Dense time series:** Every store-product combo has 90 consecutive daily data points — no sparsity
- **Supply chain relevance:** Perishable grocery demand forecasting is a classic supply chain problem
- **Built-in external regressors:** Discount, holiday, promotions, and weather features ready for MMF
- **Hierarchical structure:** City → Store → Product category hierarchy for grouped forecasting
- **Scale:** 50,000 series (sampled to 1,000 for this demo)

## 1. Setup — Install Dependencies & Create Schema

In [ ]:
%pip install datasets
dbutils.library.restartPython()

In [ ]:
# CREATE CATALOG IF NOT EXISTS mmf
# Workaround: UC validates metastore storage root before the existence check,
# so we guard with an explicit check to avoid the error when the catalog already exists.
existing_catalogs = {r.catalog for r in spark.sql("SHOW CATALOGS").collect()}
if "mmf" not in existing_catalogs:
    spark.sql("CREATE CATALOG mmf")
    print("Created catalog: mmf")
else:
    print("Catalog mmf already exists — skipping creation")

In [ ]:
# CREATE SCHEMA IF NOT EXISTS mmf.fresh_retail_net
# (Catalog mmf is expected to already exist — created manually or by cell 4)
spark.sql("CREATE SCHEMA IF NOT EXISTS mmf.fresh_retail_net")
print("Schema mmf.fresh_retail_net: ready")

## 2. Download the Dataset from HuggingFace

In [ ]:
import tempfile, os

# HuggingFace datasets needs a writable cache dir (serverless blocks /root/.cache)
hf_cache = tempfile.mkdtemp()
os.environ["HF_HOME"] = hf_cache
os.environ["HF_DATASETS_CACHE"] = hf_cache

from datasets import load_dataset

hf_ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K", cache_dir=hf_cache)
print(hf_ds)

## 3. Convert to Spark and Save Raw Data

The dataset has `hours_sale` (list of 24 floats) and `hours_stock_status` (list of 24 ints) columns.
We drop these array columns for the Delta table since MMF operates at daily granularity.

In [ ]:
import pandas as pd

# Combine train and eval splits
pdf_train = hf_ds["train"].to_pandas()
pdf_eval = hf_ds["eval"].to_pandas()
pdf_train["split"] = "train"
pdf_eval["split"] = "eval"
pdf_raw = pd.concat([pdf_train, pdf_eval], ignore_index=True)

print(f"Train rows:  {len(pdf_train):,}")
print(f"Eval rows:   {len(pdf_eval):,}")
print(f"Total rows:  {len(pdf_raw):,}")
print(f"Columns:     {list(pdf_raw.columns)}")
print(f"Date range:  {pdf_raw['dt'].min()} — {pdf_raw['dt'].max()}")
print(f"Unique stores:   {pdf_raw['store_id'].nunique():,}")
print(f"Unique products: {pdf_raw['product_id'].nunique():,}")
print(f"Unique combos:   {pdf_raw.groupby(['store_id','product_id']).ngroups:,}")

In [ ]:
# Drop hourly array columns (not needed for daily MMF) and convert date
pdf_raw = pdf_raw.drop(columns=["hours_sale", "hours_stock_status"])
pdf_raw["dt"] = pd.to_datetime(pdf_raw["dt"]).dt.date

df_raw = spark.createDataFrame(pdf_raw)
df_raw.printSchema()

In [ ]:
df_raw.write.mode("overwrite").saveAsTable("mmf.fresh_retail_net.daily_sales_raw")
print("Saved raw data to mmf.fresh_retail_net.daily_sales_raw")

## 4. Explore the Raw Data

In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT store_id) AS unique_stores,
  COUNT(DISTINCT product_id) AS unique_products,
  COUNT(DISTINCT city_id) AS unique_cities,
  MIN(dt) AS min_date,
  MAX(dt) AS max_date,
  DATEDIFF(MAX(dt), MIN(dt)) + 1 AS total_days,
  ROUND(AVG(sale_amount), 4) AS avg_daily_sales,
  ROUND(AVG(discount), 4) AS avg_discount
FROM mmf.fresh_retail_net.daily_sales_raw

In [ ]:
%sql
-- Check density: how many days does each store-product combo have?
SELECT
  MIN(day_count) AS min_days,
  ROUND(AVG(day_count), 1) AS avg_days,
  MAX(day_count) AS max_days
FROM (
  SELECT store_id, product_id, COUNT(*) AS day_count
  FROM mmf.fresh_retail_net.daily_sales_raw
  GROUP BY store_id, product_id
)

## 5. Sample to 1,000 Store-Product Combos & Prepare for MMF

We sample 1,000 random store-product combinations from the training split,
then build the MMF-ready table using only data from those combos (both train and eval splits).

In [ ]:
from pyspark.sql import functions as F

df = spark.table("mmf.fresh_retail_net.daily_sales_raw")

# Build unique_id on full dataset first
df = df.withColumn("unique_id", F.concat_ws("_", F.col("store_id"), F.col("product_id")))

# Get combos that exist in BOTH train and eval splits (required for MMF train/score alignment)
train_ids = df.filter(F.col("split") == "train").select("unique_id").distinct()
eval_ids = df.filter(F.col("split") == "eval").select("unique_id").distinct()
shared_ids = train_ids.intersect(eval_ids)
print(f"Combos in both splits: {shared_ids.count():,}")

# Deterministic sample: hash-based ordering to ensure reproducibility
sampled_ids = (
    shared_ids
    .orderBy(F.hash("unique_id"))
    .limit(1000)
)
sampled_ids.count()  # Materialize the cache
print(f"Sampled combos: {sampled_ids.count()}")

In [ ]:
# Filter to sampled combos (both train and eval data)
df_sampled = (
    df.join(sampled_ids, on="unique_id", how="inner")
    .withColumnRenamed("dt", "ds")
    .withColumnRenamed("sale_amount", "y")
)

# Verify both splits are present for all sampled series
train_count = df_sampled.filter(F.col("split") == "train").select("unique_id").distinct().count()
eval_count = df_sampled.filter(F.col("split") == "eval").select("unique_id").distinct().count()
print(f"Sampled rows: {df_sampled.count():,}")
print(f"Train series: {train_count}  |  Eval series: {eval_count}")
assert train_count == eval_count == 1000, f"Mismatch! train={train_count}, eval={eval_count}"

## 6. Split into Train & Score Tables for MMF

MMF expects two separate tables:
- **Train table** — historical data **with** the target column (`y`)
- **Score table** — future dates **without** the target column, but with external regressors

The FreshRetailNet dataset already provides a natural split:
- `split == "train"` → 90 days of history (used for training)
- `split == "eval"` → 7 days of future data (used for scoring)

In [ ]:
regressor_cols = [
    "discount",
    "holiday_flag",
    "activity_flag",
    "precpt",
    "avg_temperature",
    "avg_humidity",
    "avg_wind_level",
]

grouping_cols = [
    "city_id",
    "store_id",
    "product_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
]

# Train table: historical data WITH the target (y)
df_train = (
    df_sampled
    .filter(F.col("split") == "train")
    .select("unique_id", "ds", "y", *regressor_cols, *grouping_cols, "stock_hour6_22_cnt")
)

# Score table: future dates WITHOUT the target, but with regressors
df_score = (
    df_sampled
    .filter(F.col("split") == "eval")
    .select("unique_id", "ds", *regressor_cols, *grouping_cols)
)

print(f"Train rows: {df_train.count():,}  |  Series: {df_train.select('unique_id').distinct().count()}")
print(f"Score rows: {df_score.count():,}  |  Series: {df_score.select('unique_id').distinct().count()}")

# Verify date ranges don't overlap
train_max = df_train.agg(F.max("ds")).collect()[0][0]
score_min = df_score.agg(F.min("ds")).collect()[0][0]
print(f"\nTrain date range: {df_train.agg(F.min('ds')).collect()[0][0]} — {train_max}")
print(f"Score date range: {score_min} — {df_score.agg(F.max('ds')).collect()[0][0]}")
print(f"Gap check: train ends {train_max}, score starts {score_min} (should be consecutive)")

In [ ]:
df_train.write.mode("overwrite").saveAsTable("mmf.fresh_retail_net.demand_train")
df_score.write.mode("overwrite").saveAsTable("mmf.fresh_retail_net.demand_score")
print("Saved train data to mmf.fresh_retail_net.demand_train")
print("Saved score data to mmf.fresh_retail_net.demand_score")

## 7. Validate the Output

In [ ]:
%sql
SELECT 'train' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT unique_id) AS num_series,
  MIN(ds) AS min_date,
  MAX(ds) AS max_date,
  DATEDIFF(MAX(ds), MIN(ds)) + 1 AS total_days,
  ROUND(AVG(y), 4) AS avg_daily_demand,
  ROUND(SUM(CASE WHEN y > 0 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS pct_nonzero
FROM mmf.fresh_retail_net.demand_train
UNION ALL
SELECT 'score' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT unique_id) AS num_series,
  MIN(ds) AS min_date,
  MAX(ds) AS max_date,
  DATEDIFF(MAX(ds), MIN(ds)) + 1 AS total_days,
  NULL AS avg_daily_demand,
  NULL AS pct_nonzero
FROM mmf.fresh_retail_net.demand_score

In [ ]:
%sql
-- Top 20 series by total demand (train data)
SELECT
  unique_id,
  COUNT(*) AS total_days,
  ROUND(SUM(y), 2) AS total_demand,
  ROUND(AVG(y), 4) AS avg_daily_demand,
  SUM(CASE WHEN y > 0 THEN 1 ELSE 0 END) AS active_days
FROM mmf.fresh_retail_net.demand_train
GROUP BY unique_id
ORDER BY total_demand DESC
LIMIT 20

In [ ]:
%sql
-- Sample time series for the top-demand series (train only)
SELECT ds, y, discount, holiday_flag, activity_flag, avg_temperature
FROM mmf.fresh_retail_net.demand_train
WHERE unique_id = (
  SELECT unique_id
  FROM mmf.fresh_retail_net.demand_train
  GROUP BY unique_id
  ORDER BY SUM(y) DESC
  LIMIT 1
)
ORDER BY ds

## Output Tables

### `mmf.fresh_retail_net.demand_train` — Training data (1,000 series x 90 days)

| Column | Type | MMF Role | Description |
|---|---|---|---|
| `unique_id` | STRING | **Group ID** | `{store_id}_{product_id}` — the forecasting unit |
| `ds` | DATE | **Time column** | Date (daily, no gaps) |
| `y` | DOUBLE | **Target** | Daily sales amount (normalized) |
| `discount` | DOUBLE | Regressor | Discount rate (1.0 = no discount) |
| `holiday_flag` | INT | Regressor | Holiday indicator |
| `activity_flag` | INT | Regressor | Promotion indicator |
| `precpt` | DOUBLE | Regressor | Precipitation |
| `avg_temperature` | DOUBLE | Regressor | Average temperature |
| `avg_humidity` | DOUBLE | Regressor | Average humidity |
| `avg_wind_level` | DOUBLE | Regressor | Average wind level |
| `stock_hour6_22_cnt` | INT | Info | Out-of-stock hours (6am–10pm) |
| `city_id` | INT | Grouping | City |
| `store_id` | INT | Grouping | Store |
| `product_id` | INT | Grouping | Product |
| `management_group_id` | INT | Grouping | Management group |
| `first_category_id` | INT | Grouping | Product category L1 |
| `second_category_id` | INT | Grouping | Product category L2 |
| `third_category_id` | INT | Grouping | Product category L3 |

### `mmf.fresh_retail_net.demand_score` — Scoring data (1,000 series x 7 days)

Same columns as train **except `y` is excluded** (this is what MMF will predict).

### Example MMF `run_forecast` call:
```python
run_forecast(
    spark=spark,
    train_data="mmf.fresh_retail_net.demand_train",
    scoring_data="mmf.fresh_retail_net.demand_score",
    scoring_output="mmf.fresh_retail_net.scoring_output",
    evaluation_output="mmf.fresh_retail_net.evaluation_output",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="D",
    prediction_length=7,
    backtest_length=14,
    stride=7,
    metric="smape",
    train_predict_ratio=2,
    dynamic_future_numerical=["discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"],
    dynamic_future_categorical=["holiday_flag", "activity_flag"],
)
```